## 2. Proportional harvesting and recovery

<div style="border-left:5px solid #c87116;background:#c8711614;padding:10px 14px"><strong>Question:</strong> Does reducing the catch restore a collapsed population? Carry state through the whole experiment and distinguish recovery from explicit restocking.</div>

Owner: @AshChan07. These are synthetic model calculations, not observations.

**Question:** does a smaller catch when population declines guarantee recovery?
No: below the Allee threshold, natural growth is already negative.
At 100 tonnes an effort of 0.05/year removes 5 tonnes/year, but at 20 tonnes
it removes 1 tonne/year. We compare this with a constant catch using common units.

## Derivation before computation
With $x=N/K$, $\tau=rt$, $a=A/K$, $e=h/r$,
$$f_B(x)=x[(1-x)(x/a-1)-e].$$
The nonzero roots solve $x^2-(1+a)x+a(1+e)=0$, so
$$x_\pm=\frac{1+a\pm\sqrt{(1-a)^2-4ae}}2,\qquad e_c=\frac{(1-a)^2}{4a}.$$
For $e<e_c$, $f_B=-x(x-x_-)(x-x_+)/a$ has signs $-,+,-$ on the three positive intervals.
Thus the lower positive root is unstable (the basin boundary) and the upper root is stable.
Zero is stable because $f'_B(0)=-1-e<0$. At the fold, the factor is a square:
the field is negative on both sides, so attraction occurs only from above.
Above the threshold only zero remains. At zero effort, recover $0,a,1$.

**Algorithm:** compute the discriminant threshold; return zero; return no positive
roots above it, one double root at it, or two roots below it. Use the product of
roots for the smaller root to avoid cancellation. The `e` field is effort, not
Task 1's yield `q`. Independent high-precision polynomial and trajectory tests
check this implementation.

In [ ]:
import csv
import json
import os
from pathlib import Path

from IPython.display import Image, display

from mm_allee.model_b import critical_effort, equilibria_b
from mm_allee.wp02_experiment import run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
for effort in [0, 0.4, critical_effort(0.2), 1.0]:
    print(effort, [(r.x, r.stability) for r in equilibria_b(0.2, effort)])

## Fair comparison definitions

1. **Initial catch:** $H=hN_0$. Set $H=\alpha H_c^A$ and solve for $h$; the logistic model also gets $H$.
2. **Threshold fraction:** match $H_A/H_c^A=h/h_c=H_L/H_c^L=\alpha$. Initial catches can differ.

Use $r=0.5$/year, $K=1000$ tonnes, $A=200$ tonnes; $N_0/K=0.15,0.4,0.9$ and $\alpha=0.5,1.2$.
Calculate theoretical outcomes from phase lines separately from the finite observation horizon ($\tau=30$).
Integrate actual B catch over time; stop A/logistic catch at extinction. Divide total catch by the full horizon.
A blank critical population indicates no populated equilibrium. Equal $r,K$ does not match
logistic and Allee production curves, so a universal ecological risk ranking would be unjustified.

## History algorithm

Use staircase controls from 0 to 1.3 times critical and back, with 12 increments each way,
12 units at the peak, and dwell times 1 and 3 on ordinary intervals. Carry every terminal state forward.
After returning to zero, wait 15 units, then compare a 20-unit unassisted hold with a run adding $0.6K$.
For A, record the first zero event and shut down harvest; the zero continuation is a declared policy.
For B, never use the cutoff to reset the state. Report only the first sampled $x\leq10^{-6}$,
not an exact extinction time. Paired boundary rows encode left/right states; only restocking changes population discontinuously.

The following executes the actual bounded CPU experiment and writes a manifest. It also reruns
all four restocking histories with tighter integration settings; errors must be below the
predeclared absolute limits, $2\times10^{-7}$ in population and event time.

In [ ]:
config = json.loads((ROOT / "configs/wp02.json").read_text())
output = ROOT / "results/wp02-notebook"
manifest = run_experiment(config, output, ROOT)
print(manifest["status"], manifest["elapsed_seconds"], "seconds")
display(Image(filename=str(output / "wp02_b_branches.png")))
display(Image(filename=str(output / "wp02_history.png")))

In [ ]:
with (output / "wp02_comparison.csv").open() as f:
    comparison = list(csv.DictReader(f))
for row in comparison:
    if float(row["initial_population"]) == 900 and float(row["fraction"]) == 0.5:
        print(
            row["protocol"],
            row["model"],
            "initial catch",
            round(float(row["initial_catch"]), 2),
            "final biomass",
            round(float(row["final_population"]), 2),
            row["theoretical_outcome"],
        )
for row in manifest["summary"]:
    print(row)

## Interpretation and limits

At equal initial catch (131.28 tonnes/year), the example from 900 tonnes persists for A/B
but collapses for the logistic model: the catch exceeds that model's 125 tonnes/year maximum.
At half of each threshold, all three persist from 900 tonnes, but catches differ.
This contrast is about calibration, not evidence that Allee growth always improves safety.

For both ramp speeds, unassisted A reaches extinction and B becomes very small.
Reducing harvest to zero fails to restore a population below $a=0.2$.
Adding $0.6K$ at zero harvest crosses this threshold and restores the population toward $K$.
This is history dependence with an intervention, **not a spontaneous closed hysteresis loop**.
A event times in the curated run are about 13.436601 and 34.612578 in $\tau$ units;
B's first sampled smallness times are 20.4 and 42.0. Dynamic delay is not a new equilibrium threshold.
The two schedules also change total exposure, not just ramp speed.

Refinement errors in the curated runs are below $2\times10^{-9}$; tiny B tails below solver
absolute tolerance have no claimed relative accuracy. Test checks include high-precision roots,
independent maximisation, perturbations, matched units, boundary continuity, insufficient and sufficient
restocking, numerical refinement and manifest checksums. See `docs/tasks/WP02_model_b.md` for the evidence map.

The integrated notebook uses current reviewed WP01 code and WP03's shared threshold/basin helpers.
Comparisons retain the validated range $0.05\leq a\leq0.95$ and conservatively exclude ambiguous near-fold inputs.
A near-boundary initial population is explicitly unresolved. Final group slides, human reviews and rehearsal remain pending.
Optional slowing-down experiments are deferred. Codex assisted drafting and implementation;
human understanding has not been certified.

**Presentation check:** why does zero remain stable when effort falls back to zero?
Explain the sign of natural growth below the Allee threshold before quoting a formula.